In [3]:
import pandas as pd
import os
from datetime import datetime, timedelta

directory = 'C:/Users/tuan-/Downloads/1 Lobster Thesis/Data/SPY2018/'

In [8]:
# Konverter tid
def convert_to_datetime(seconds, base_date):
    base_time = datetime.strptime(base_date, '%Y-%m-%d')
    return base_time + timedelta(seconds=seconds)

monthly_data = []

year = 2018

for month in range(1, 13):  # Loop monthly 
    monthly_files = []
    
    for filename in os.listdir(directory):
        if f'{year}-{month:02}' in filename and 'message' in filename:   # Match filer monthly and year
            message_file = os.path.join(directory, filename)
            orderbook_file = message_file.replace('message', 'orderbook')  # Match orderbook fil
            
             # Load message and orderbook 
            message_df = pd.read_csv(message_file)
            orderbook_df = pd.read_csv(orderbook_file)

            # Message and orderbook data, dropper col 7
            message_df = message_df.iloc[:, :-1]  
            message_df.columns = ['Time (sec)', 'Event Type', 'Order ID', 'Size', 'Price', 'Direction']
            orderbook_df.columns = ['Ask Price 1', 'Ask Size 1', 'Bid Price 1', 'Bid Size 1', 
                                    'Ask Price 2', 'Ask Size 2', 'Bid Price 2', 'Bid Size 2']

            base_date = filename.split('_')[1]  # Start dag SPY_2018-01-02
            message_df['Time (sec)'] = message_df['Time (sec)'].apply(lambda x: convert_to_datetime(x, base_date))

            # Merger message and orderbook data on 'Time (sec)'
            combined_df = pd.merge(orderbook_df, message_df, left_index=True, right_index=True, how='left')

            # Drop NAN
            combined_df.dropna(inplace=True)

            # Set 'Time (sec)' index for 1 second
            combined_df.set_index('Time (sec)', inplace=True)

            # Sampler data 1 sec interval første obs hvert sec
            resampled_df = combined_df.resample('1S').first()

            # list monthly
            monthly_files.append(resampled_df)
    
    # Concatenate all daily filer for month
    if monthly_files:
        monthly_data_df = pd.concat(monthly_files)
        monthly_data.append(monthly_data_df)
        
        # Gem
        monthly_data_df.to_csv(f'processed_{year}_{month:02}.csv', index=False)

# All months into one final DataFrame
final_df = pd.concat(monthly_data)

# Save the final DataFrame, year 2018
final_df.to_csv(f'combined_SPY{year}_cleaned.csv', index=False)


print(final_df.head())

In [6]:
print(final_df.head())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2018-01-02 09:30:00    2678500.0       100.0    2678400.0       800.0   
2018-01-02 09:30:01    2678900.0       910.0    2678700.0        15.0   
2018-01-02 09:30:02    2678400.0      1000.0    2678000.0      1100.0   
2018-01-02 09:30:03    2678600.0      5600.0    2678500.0      2200.0   
2018-01-02 09:30:04    2678200.0      4085.0    2678100.0      1500.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2018-01-02 09:30:00    2678700.0      1100.0    2678300.0      1000.0   
2018-01-02 09:30:01    2679000.0      4000.0    2678600.0      1100.0   
2018-01-02 09:30:02    2678500.0      1100.0    2677900.0       100.0   
2018-01-02 09:30:03    2678700.0      5000.0    2678400.0      1000.0   
2018-01-02 09:30:04    2678400.0      1000.0    26

In [9]:
print(final_df.tail())

                     Ask Price 1  Ask Size 1  Bid Price 1  Bid Size 1  \
Time (sec)                                                              
2018-12-31 15:59:55    2498400.0       129.0    2498200.0       100.0   
2018-12-31 15:59:56    2498600.0       200.0    2498500.0       171.0   
2018-12-31 15:59:57    2499100.0       500.0    2498900.0      1400.0   
2018-12-31 15:59:58    2499000.0       300.0    2498800.0       800.0   
2018-12-31 15:59:59    2499700.0      6800.0    2499500.0      2700.0   

                     Ask Price 2  Ask Size 2  Bid Price 2  Bid Size 2  \
Time (sec)                                                              
2018-12-31 15:59:55    2498500.0      7600.0    2498100.0       400.0   
2018-12-31 15:59:56    2498700.0       700.0    2498400.0       800.0   
2018-12-31 15:59:57    2499200.0       500.0    2498800.0       600.0   
2018-12-31 15:59:58    2499100.0       300.0    2498700.0      1100.0   
2018-12-31 15:59:59    2499800.0      7400.0    24

In [10]:
# Observations (rows) in the final combined DataFrame for the year
total_observations_final = len(final_df)

print(f"Total observations in the final combined DataFrame: {total_observations_final}")

Total observations in the final combined DataFrame: 5873396


In [11]:
final_df.head()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction
Time (sec),,,,,,,,,,,,,
2018-01-02 09:30:00,2678500.0,100.0,2678400.0,800.0,2678700.0,1100.0,2678300.0,1000.0,3.0,6289088.0,1000.0,2678400.0,1.0
2018-01-02 09:30:01,2678900.0,910.0,2678700.0,15.0,2679000.0,4000.0,2678600.0,1100.0,4.0,6524680.0,100.0,2678800.0,-1.0
2018-01-02 09:30:02,2678400.0,1000.0,2678000.0,1100.0,2678500.0,1100.0,2677900.0,100.0,1.0,6662404.0,1000.0,2678000.0,1.0
2018-01-02 09:30:03,2678600.0,5600.0,2678500.0,2200.0,2678700.0,5000.0,2678400.0,1000.0,1.0,6805984.0,1000.0,2678600.0,-1.0
2018-01-02 09:30:04,2678200.0,4085.0,2678100.0,1500.0,2678400.0,1000.0,2678000.0,20.0,1.0,6872116.0,100.0,2678100.0,1.0


In [12]:
final_df.tail()

,Ask Price 1,Ask Size 1,Bid Price 1,Bid Size 1,Ask Price 2,Ask Size 2,Bid Price 2,Bid Size 2,Event Type,Order ID,Size,Price,Direction
Time (sec),,,,,,,,,,,,,
2018-12-31 15:59:55,2498400.0,129.0,2498200.0,100.0,2498500.0,7600.0,2498100.0,400.0,1.0,351211280.0,29.0,2498400.0,-1.0
2018-12-31 15:59:56,2498600.0,200.0,2498500.0,171.0,2498700.0,700.0,2498400.0,800.0,4.0,351293004.0,29.0,2498500.0,1.0
2018-12-31 15:59:57,2499100.0,500.0,2498900.0,1400.0,2499200.0,500.0,2498800.0,600.0,1.0,351374012.0,1100.0,2498900.0,1.0
2018-12-31 15:59:58,2499000.0,300.0,2498800.0,800.0,2499100.0,300.0,2498700.0,1100.0,1.0,351451792.0,500.0,2498800.0,1.0
2018-12-31 15:59:59,2499700.0,6800.0,2499500.0,2700.0,2499800.0,7400.0,2499400.0,400.0,4.0,351528412.0,100.0,2499500.0,1.0


In [14]:
# Save 
final_df.to_csv('final_combined_2018.csv', index=True)  # Keep the index to retain 'Time (sec)'

final_df.to_csv('final_combined_2018_with_time.csv', index=True)

In [15]:
# Count unique days
final_df['Date'] = final_df.index.date

# Count the number of unique trading days
unique_trading_days = final_df['Date'].nunique()

print(f"Number of unique trading days: {unique_trading_days}")

Number of unique trading days: 251


In [16]:
# Load the data from 'final_combined_2018_with_time.csv'
final_combined_2018_with_time = pd.read_csv('final_combined_2018_with_time.csv')


final_combined_2018_with_time['Time (sec)'] = pd.to_datetime(final_combined_2018_with_time['Time (sec)'])

# Group trading days
grouped = final_combined_2018_with_time.groupby(final_combined_2018_with_time['Time (sec)'].dt.date)

# Sampler liste
resampled_5min_list = []

# Iterate through each group each day
for date, group in grouped:
    # Filtrer data for trading hours (9:30 AM to 4:00 PM)
    group_trading_hours = group[
        (group['Time (sec)'].dt.time >= pd.to_datetime('09:30:00').time()) &
        (group['Time (sec)'].dt.time <= pd.to_datetime('16:00:00').time())
    ]
    
    # Resample for 5-minute intervals 
    resampled_day = group_trading_hours.resample('5T', on='Time (sec)').agg({
        'Ask Price 1': ['first', 'max', 'min', 'last'],
        'Bid Price 1': ['first', 'max', 'min', 'last'],
        'Ask Price 2': ['first', 'max', 'min', 'last'],
        'Bid Price 2': ['first', 'max', 'min', 'last'],
        'Ask Size 1': ['sum', 'mean'],
        'Bid Size 1': ['sum', 'mean'],
        'Ask Size 2': ['sum', 'mean'],
        'Bid Size 2': ['sum', 'mean'],
        'Price': ['first', 'max', 'min', 'last'],
        'Direction': 'mean'
    })
    
   
    resampled_day.columns = ['_'.join(col).strip() for col in resampled_day.columns.values]
    
    # Append the resampled data dag
    resampled_5min_list.append(resampled_day)

# Concatenate all, DataFrame
resampled_5min_final_df = pd.concat(resampled_5min_list)

# Reset index 'Time (sec)' 
resampled_5min_final_df.reset_index(inplace=True)

In [17]:
print(resampled_5min_final_df.head(10))

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2018-01-02 09:30:00          2678500.0        2678900.0        2674700.0   
1 2018-01-02 09:35:00          2674800.0        2678200.0        2674000.0   
2 2018-01-02 09:40:00          2678000.0        2679300.0        2677800.0   
3 2018-01-02 09:45:00          2678200.0        2678400.0        2676800.0   
4 2018-01-02 09:50:00          2676900.0        2677600.0        2676100.0   
5 2018-01-02 09:55:00          2677300.0        2680000.0        2676600.0   
6 2018-01-02 10:00:00          2679800.0        2682200.0        2679700.0   
7 2018-01-02 10:05:00          2681100.0        2682700.0        2681100.0   
8 2018-01-02 10:10:00          2682600.0        2682600.0        2681600.0   
9 2018-01-02 10:15:00          2681700.0        2683000.0        2681700.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         2674700.0          2678400.0        2678700.0        26

In [22]:
# Save
resampled_5min_final_df.to_csv('resampled_5min_final_2018_corrected.csv', index=False)

In [19]:
# Count observations for each day in 5-minute interval
resampled_5min_final_df['Date'] = resampled_5min_final_df['Time (sec)'].dt.date

# Count the number of rows for each day
observations_per_day = resampled_5min_final_df.groupby('Date').size()

print(observations_per_day)

Date
2018-01-02    78
2018-01-03    78
2018-01-04    78
2018-01-05    78
2018-01-08    78
              ..
2018-12-24    78
2018-12-26    78
2018-12-27    78
2018-12-28    78
2018-12-31    78
Length: 251, dtype: int64


In [23]:
print(resampled_5min_final_df.head(10))

# Save igen
resampled_5min_final_df.to_csv('resampled_5min_final_2018_corrected.csv', index=False)

           Time (sec)  Ask Price 1_first  Ask Price 1_max  Ask Price 1_min  \
0 2018-01-02 09:30:00          2678500.0        2678900.0        2674700.0   
1 2018-01-02 09:35:00          2674800.0        2678200.0        2674000.0   
2 2018-01-02 09:40:00          2678000.0        2679300.0        2677800.0   
3 2018-01-02 09:45:00          2678200.0        2678400.0        2676800.0   
4 2018-01-02 09:50:00          2676900.0        2677600.0        2676100.0   
5 2018-01-02 09:55:00          2677300.0        2680000.0        2676600.0   
6 2018-01-02 10:00:00          2679800.0        2682200.0        2679700.0   
7 2018-01-02 10:05:00          2681100.0        2682700.0        2681100.0   
8 2018-01-02 10:10:00          2682600.0        2682600.0        2681600.0   
9 2018-01-02 10:15:00          2681700.0        2683000.0        2681700.0   

   Ask Price 1_last  Bid Price 1_first  Bid Price 1_max  Bid Price 1_min  \
0         2674700.0          2678400.0        2678700.0        26

In [24]:
from IPython.display import display, HTML

# Display as an HTML tabel
html_table = resampled_5min_final_df.head(10).to_html(index=False)
display(HTML(html_table))

# Gem denne
resampled_5min_final_df.to_csv('resampled_5min_final_2018_corrected.csv', index=False)

Time (sec),Ask Price 1_first,Ask Price 1_max,Ask Price 1_min,Ask Price 1_last,Bid Price 1_first,Bid Price 1_max,Bid Price 1_min,Bid Price 1_last,Ask Price 2_first,Ask Price 2_max,Ask Price 2_min,Ask Price 2_last,Bid Price 2_first,Bid Price 2_max,Bid Price 2_min,Bid Price 2_last,Ask Size 1_sum,Ask Size 1_mean,Bid Size 1_sum,Bid Size 1_mean,Ask Size 2_sum,Ask Size 2_mean,Bid Size 2_sum,Bid Size 2_mean,Price_first,Price_max,Price_min,Price_last,Direction_mean,Date
2018-01-02 09:30:00,2678500.0,2678900.0,2674700.0,2674700.0,2678400.0,2678700.0,2674600.0,2674600.0,2678700.0,2679000.0,2674800.0,2674800.0,2678300.0,2678600.0,2674500.0,2674500.0,496836.0,1667.234899,412101.0,1382.889262,690137.0,2315.895973,511151.0,1715.271812,2678400.0,2678800.0,2674500.0,2674600.0,0.147651,2018-01-02
2018-01-02 09:35:00,2674800.0,2678200.0,2674000.0,2678000.0,2674700.0,2678000.0,2673900.0,2677900.0,2674900.0,2678300.0,2674100.0,2678100.0,2674600.0,2677900.0,2673800.0,2677800.0,354005.0,1183.963211,402219.0,1345.214047,534500.0,1787.625418,503314.0,1683.324415,2674700.0,2678000.0,2673900.0,2677900.0,0.150502,2018-01-02
2018-01-02 09:40:00,2678000.0,2679300.0,2677800.0,2678300.0,2677900.0,2679200.0,2677700.0,2678100.0,2678100.0,2679400.0,2677900.0,2678400.0,2677800.0,2679100.0,2677600.0,2678000.0,534248.0,1804.891892,564348.0,1906.581081,762773.0,2576.935811,712397.0,2406.746622,2677900.0,2679300.0,2677700.0,2678100.0,-0.047297,2018-01-02
2018-01-02 09:45:00,2678200.0,2678400.0,2676800.0,2676900.0,2678100.0,2678300.0,2676700.0,2676800.0,2678300.0,2678500.0,2676900.0,2677000.0,2678000.0,2678200.0,2676600.0,2676700.0,490243.0,1645.110738,521277.0,1749.251678,587848.0,1972.644295,634699.0,2129.862416,2678200.0,2678400.0,2676700.0,2676800.0,0.033557,2018-01-02
2018-01-02 09:50:00,2676900.0,2677600.0,2676100.0,2677300.0,2676700.0,2677500.0,2676000.0,2677200.0,2677000.0,2677700.0,2676200.0,2677400.0,2676600.0,2677400.0,2675900.0,2677100.0,394136.0,1331.540541,447317.0,1511.206081,497786.0,1681.709459,725004.0,2449.337838,2676800.0,2677600.0,2676100.0,2677200.0,-0.168919,2018-01-02
2018-01-02 09:55:00,2677300.0,2680000.0,2676600.0,2679800.0,2677200.0,2679900.0,2676500.0,2679700.0,2677400.0,2680100.0,2676700.0,2679900.0,2677100.0,2679800.0,2676400.0,2679600.0,452172.0,1532.786441,542107.0,1837.650847,629943.0,2135.400000,704176.0,2387.037288,2677300.0,2680000.0,2676600.0,2679700.0,-0.227119,2018-01-02
2018-01-02 10:00:00,2679800.0,2682200.0,2679700.0,2681200.0,2679700.0,2682100.0,2679600.0,2681100.0,2679900.0,2682300.0,2679800.0,2681300.0,2679600.0,2682000.0,2679500.0,2681000.0,535464.0,1809.000000,495729.0,1674.760135,548364.0,1852.581081,617033.0,2084.570946,2679900.0,2682200.0,2679700.0,2681200.0,-0.155405,2018-01-02
2018-01-02 10:05:00,2681100.0,2682700.0,2681100.0,2682600.0,2681000.0,2682600.0,2681000.0,2682400.0,2681200.0,2682800.0,2681200.0,2682700.0,2680900.0,2682500.0,2680900.0,2682300.0,379794.0,1332.610526,511682.0,1795.375439,523066.0,1835.319298,651113.0,2284.607018,2681000.0,2682800.0,2681000.0,2682600.0,-0.178947,2018-01-02
2018-01-02 10:10:00,2682600.0,2682600.0,2681600.0,2681700.0,2682400.0,2682400.0,2681500.0,2681600.0,2682700.0,2682700.0,2681700.0,2681800.0,2682300.0,2682300.0,2681400.0,2681500.0,449379.0,1670.553903,503286.0,1870.951673,655974.0,2438.565056,660592.0,2455.732342,2682400.0,2682400.0,2681400.0,2681600.0,-0.345725,2018-01-02
2018-01-02 10:15:00,2681700.0,2683000.0,2681700.0,2682500.0,2681600.0,2682900.0,2681600.0,2682400.0,2681800.0,2683100.0,2681800.0,2682600.0,2681500.0,2682800.0,2681500.0,2682300.0,378230.0,1350.821429,485732.0,1734.757143,670089.0,2393.175000,596339.0,2129.782143,2681700.0,2683000.0,2681700.0,2682400.0,-0.264286,2018-01-02
